In [2]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns

import sklearn
sklearn.set_config(display='text')
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

from sklearn.ensemble import AdaBoostClassifier # 앙상블 에이다 부스트 알고리즘을 사용하기 위해 import 한다.

부스팅(boosting)

초기에는 모든 데이터 포인트에 동일한 가중치를 할당하고 점차 학습이 진행되면서 올바르게 분류된 데이터 포인트 가중치는 감소시키는 반면에 잘못 분류된 데이터 포인트의 가중치는 증가시킨다.  
결과적으로 학습이 진행되면서 학습기는 잘못 분류된(분류하기 어려운) 데이터에 집중하게 되고 이전 단계에서 만들어진 학습기는 다음 단계에서 사용할 학습 데이터의 가중치를 반영하는데 사용하므로 부스팅은 배깅되 달리 이전 분류기의 영향을 받는다.

배깅은 각 데이터 포인트가 추출될 확률이 동일하지만, 부스팅은 각 데이터 포인트에 할당된 가중치에 비례해서 추출된다.

에이다 부스트(adaboost)

약한 학습기를 여러 개를 모아서 하나의 강한 학습기를 만드는 방법으로 개별적으로는 약한 학습 모델이지만, 이와 같은 모델을 다수 생성하고 부스팅을 적용함으로써 강한 학습기가 만들어진다.

보팅이나 배깅은 모델이 병렬적으로 실행된다. 10개의 모델이 있다면 10개의 모델을 동시에 학습시킬 수 있다는 뜻이다. 이에 반해 부스팅은 여러 개의 약한 학습기가 순차적으로 적용된다. 그 이유는 약한 학습 모델의 학습 이후 판별하지 못한 데이터 포인트에 대해서 가중를 부여하기 때문이다.

에이다 부스트의 핵심 아이디어는 분류하기 어려운 학습 데이터에 가중치를 더 높이는 것이다. 즉, 이전에 잘못 분류된 학습 데이터 포인트는 가중치가 증가해 오차율이 높아진다. 약한 학습기는 이전 학습기에서 증가한 오차를 낮추는 방향으로 학습하게 된다.  
에이다 부스트는 일반적인 부스팅과는 다르게 약한 학습기로 학습할 때 학습 데이터셋 전체를 사용한다. 학습 샘플은 반복할 때 마다 가중치가 부여되며 이 앙상블은 이전 학습기가 실수한 부분을 학습하는 강력한 분류를 만든다.

와인 데이터를 사용해서 와인 종류를 분류하는 모델을 생성하고 학습시킨다.

In [4]:
# 데이터 불러오기
raw_data = datasets.load_wine() # 사이킷런 라이브러리가 제공하는 와인 데이터를 불러온다.
# print(raw_data)

# 피쳐, 레이블 데이터 저장
xData = raw_data.data # 피쳐 데이터를 저장한다.
yData = raw_data.target # 피쳐 데이터에 따른 레이블을 저장한다.
# print(xData.shape, yData.shape)

# 학습 데이터와 테스트 데이터로 분할
x_train, x_test, y_train, y_test = train_test_split(xData, yData, random_state=0)
# print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

# 데이터 표준화(정규화)
scaler = StandardScaler() # 표준화 스케일러 객체를 만든다.
x_train = scaler.fit_transform(x_train) # 학습 데이터를 표준화 스케일러로 표준화하고 적용한다.
x_test = scaler.transform(x_test) # 테스트 데이터를 학습 데이터로 표준화한 스케일러에 적용한다.

# 모델 생성 후 데이터 학습
model = AdaBoostClassifier().fit(x_train, y_train) # 앙상블 에이다 부스트 모델을 만들고 학습시킨다.

학습된 모델로 테스트 데이터를 예측한다.

In [5]:
predict = model.predict(x_test) # predict() 메소드의 인수로 표준화된 테스트 데이터(x_test)를 넘겨서 서포트 벡터 머신 모델을 예측한다.
print(predict)

[0 2 1 0 1 0 0 2 1 1 2 2 0 1 2 1 0 0 2 0 0 1 0 1 1 1 1 1 1 2 0 0 1 0 0 0 2
 1 1 2 0 0 1 1 1]


In [6]:
predict_proba = model.predict_proba(x_test) # predict_proba() 메소드의 인수로 표준화된 테스트 데이터(x_test)를 넘겨서 각 클래스에 속할 확률로 예측한다.
print(predict_proba)

[[0.39362319 0.33137377 0.27500304]
 [0.27707833 0.33073379 0.39218788]
 [0.35648921 0.37772695 0.26578384]
 [0.39362319 0.33137377 0.27500304]
 [0.35146529 0.37826057 0.27027414]
 [0.36425681 0.35580726 0.27993593]
 [0.39990877 0.32540777 0.27468346]
 [0.26769982 0.3254993  0.40680087]
 [0.31783289 0.41411464 0.26805247]
 [0.2552072  0.40063398 0.34415882]
 [0.29174064 0.30857597 0.39968339]
 [0.28106205 0.31130065 0.4076373 ]
 [0.39362319 0.33137377 0.27500304]
 [0.35648921 0.37772695 0.26578384]
 [0.2902115  0.30325544 0.40653305]
 [0.33358073 0.40761364 0.25880563]
 [0.39990877 0.32540777 0.27468346]
 [0.39362319 0.33137377 0.27500304]
 [0.2688044  0.35321331 0.37798229]
 [0.39362319 0.33137377 0.27500304]
 [0.3747431  0.34851989 0.27673702]
 [0.33843914 0.35788844 0.30367242]
 [0.38493147 0.35051532 0.26455322]
 [0.31783289 0.41411464 0.26805247]
 [0.25962371 0.38454896 0.35582733]
 [0.33425945 0.40272873 0.26301183]
 [0.35779521 0.36555568 0.27664911]
 [0.27578333 0.41624914 0.30

학습된 모델을 평가한다.

In [8]:
# 혼동 행렬
# confusion_matrix() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 혼동 행렬을 출력한다.
confusion = confusion_matrix(y_test, predict)
print(confusion)

[[15  1  0]
 [ 2 18  1]
 [ 0  0  8]]


In [9]:
# 분류 리포트
# classification_report() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 분류 리포트를 출력한다.
classification = classification_report(y_test, predict, target_names=raw_data.target_names)
print(classification)

              precision    recall  f1-score   support

     class_0       0.88      0.94      0.91        16
     class_1       0.95      0.86      0.90        21
     class_2       0.89      1.00      0.94         8

    accuracy                           0.91        45
   macro avg       0.91      0.93      0.92        45
weighted avg       0.91      0.91      0.91        45



In [10]:
# 정확도 평가
# accuracy_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 정확도를 계산한다.
accuracy = accuracy_score(y_test, predict)
print(accuracy)

0.9111111111111111


In [11]:
# 정밀도 평가
# precision_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 정밀도를 계산한다.
precision = precision_score(y_test, predict, average=None)
print(precision)

[0.88235294 0.94736842 0.88888889]


In [12]:
# 재현율 평가
# recall_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 재현율을 계산한다.
recall = recall_score(y_test, predict, average=None)
print(recall)

[0.9375     0.85714286 1.        ]


In [13]:
# f1 score 평가
# f1_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 f1 score를 계산한다.
f1 = f1_score(y_test, predict, average=None)
print(f1)

[0.90909091 0.9        0.94117647]
